## Open notebook in:
| Colab                                 
:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH10/ch10_art-e-langgraph.ipynb)                                             

In this notebook, you will be using [ART](https://github.com/openpipe/art) together with [LangGraph](https://langchain-ai.github.io/langgraph/) to train your own ART•E agent from scratch! This implementation demonstrates how to integrate LangGraph's agent framework with ART's training capabilities.

Beginning with a Qwen 2.5 7B base model, you will train it to search through emails and answer questions about them using LangGraph's ReAct agent pattern. You will construct an [agentic environment](#Environment), define a [rollout](#Rollout) using LangGraph, and run a [training loop](#Loop). You will also learn how to use [RULER](#ruler) to judge the quality of the agent's answers.

**RULER**

RULER is a robust technique for evaluating the quality of an agent's answers and training the agent to produce more of its best completions. To learn more about RULER, see the [RULER documentation](https://art.openpipe.ai/fundamentals/ruler).


<br>

__Note: Code is partially adapted from Openpipe ART example Notebooks__

<font color="red" size="6">
<b>ATT: There is a dependencies problem with Colab:</b>
</font>

<br>
<font color="black" size="4">
<b>To resolve this, go to: Runtime -> Change Runtime and then select:
Runtime version 2026.04</b>
</font>

In [1]:
!uv pip install "openpipe-art[backend]==0.5.4"

Using Python 3.12.13 environment at: /usr
Resolved 242 packages in 2.11s
Prepared 91 packages in 41.72s
Uninstalled 33 packages in 4.16s
Installed 91 packages in 410ms
 + abnf==2.2.0
 - accelerate==1.13.0
 + accelerate==1.7.0
 + astor==0.8.1
 + awscli==1.46.0
 + backoff==2.2.1
 + bitsandbytes==0.50.0
 + blake3==1.0.9
 + cbor2==6.1.4
 + cint==1.0.0
 + colorama==0.4.6
 + compressed-tensors==0.10.2
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==5.0.1
 + depyf==0.19.0
 + detect-installer==0.1.0
 + diskcache==5.6.3
 + diskcache-weave==5.6.3.post1
 + dnspython==2.8.0
 - docutils==0.21.2
 + docutils==0.19
 + email-validator==2.3.0
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.23.0
 + fastar==0.11.0
 + fickling==0.1.12
 + gguf==0.19.0
 + gql==3.5.3
 + graphql-core==3.2.6
 + hf-transfer==0.1.9
 - huggingface-hub==1.8.0
 + huggingface-hub==0.36.2
 + interegular==0.3.3
 + intervaltree==3.2.1
 + jedi==0.20.0
 + jmespath==1.1.0
 + kaitaistruct==0.11
 - lark==1.3.1
 + lark==1.2.2
 + l

In [2]:
!pip install gql==4.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: gql
    Found existing installation: gql 3.5.3
    Uninstalling gql-3.5.3:
      Successfully uninstalled gql-3.5.3


<a name="Environment-Variables"></a>

### Environment Variables

**OpenAI (used for RULER judge model)**

Our RULER reward function queries third-party models to judge the quality of the agent's performance. Any model supported by LiteLLM works. For this example we'll use OpenAI's o4-mini model, so we'll need to set the `OPENAI_API_KEY` environment variable.

**Weights & Biases (optional)**

Later on in the notebook, we'll be creating a model that can automatically logs metrics to Weights & Biases and chat completions to Weave. In order to do so, you'll need to provide your Weights & Biases API key as an environment variable.

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()


OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
WANDB_API_KEY = os.getenv('WANDB_API_KEY')


# Market Environment (tools + helpers)

In [3]:
import re, uuid, ast, operator as op
from dataclasses import dataclass
from textwrap import dedent
from typing import List, Optional, Any

import pandas as pd
import yfinance as yf

import art
from art.local import LocalBackend
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.prebuilt import create_react_agent
import weave

# ========== Robust calculator (no ast.Num deprecation) ==========
_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.USub: op.neg, ast.UAdd: op.pos
}
def _eval_expr(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy
        return node.n
    if isinstance(node, ast.UnaryOp):
        return _ALLOWED_OPS[type(node.op)](_eval_expr(node.operand))
    if isinstance(node, ast.BinOp):
        return _ALLOWED_OPS[type(node.op)](_eval_expr(node.left), _eval_expr(node.right))
    if isinstance(node, ast.Expr):
        return _eval_expr(node.value)
    raise ValueError("Unsupported expression")

@tool
def calculator(expression: str) -> float:
    """Safely evaluate a numeric expression, e.g. '(x+y+z)/3'."""
    tree = ast.parse(expression, mode="eval")
    return float(_eval_expr(tree.body))

# ========== Data fetching ==========
def fetch_prices(ticker: str, period_days: int = 10) -> pd.DataFrame:
    df = yf.Ticker(ticker).history(period=f"{period_days}d", interval="1d")
    if df is None or len(df) == 0:
        raise ValueError(f"No data returned for {ticker}.")
    out = df.reset_index()[["Date", "Close"]]
    out["Date"] = pd.to_datetime(out["Date"])
    return out

@tool
def latest_closes_tool(ticker: str, n: int = 3, period_days: int = 10) -> List[float]:
    """Return the most recent n CLOSE prices (newest first)."""
    df = fetch_prices(ticker, period_days)
    closes = df["Close"].tail(n).tolist()
    return list(reversed(closes))  # newest first

# ========== Final answer holder ==========
from pydantic import BaseModel, Field

class FinalAnswer(BaseModel):
    answer: str
    source_ids: List[str]

nonlocal_final = {"value": None}

@tool
def return_final_answer_tool(answer: str, reference_ids: List[str]) -> dict:
    """Return final numeric answer and source ids."""
    nonlocal_final["value"] = FinalAnswer(answer=answer, source_ids=reference_ids)
    return nonlocal_final["value"].model_dump()

# ========== Numeric judge (deterministic) ==========
def _round2(x: float) -> str:
    return f"{x:.2f}"

def judge_sma_numeric(ticker: str, window: int, period_days: int, y_pred: str, tol: float=0.02) -> tuple[float, float, float]:
    """Returns (reward, truth, pred) with absolute error tolerance."""
    df = fetch_prices(ticker, period_days)
    closes = df["Close"].tail(window).tolist()
    if len(closes) < window:
        return 0.0, float("nan"), float("nan")
    truth = sum(closes)/window
    try:
        pred = float(re.findall(r"-?\d+(?:\.\d+)?", y_pred)[0])
    except Exception:
        return 0.0, truth, float("nan")
    err = abs(pred - round(truth, 2))
    reward = 1.0 if err <= tol else 0.0
    return reward, truth, pred


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [ ]:
"""
# Clean reinstall of Pillow to resolve 'cannot import name _Ink'
!uv pip uninstall -y pillow pillow-core
!uv pip install --upgrade --force-reinstall "pillow==10.4.0"

import PIL, sys
print("Pillow version:", PIL.__version__)
print(sys.executable)
"""

# Model + Backend

In [4]:
import unsloth
import art
from art.local import LocalBackend
import random


random.seed(42)

# Declare the model
model = art.TrainableModel(
    name="market-sma-langgraph-002",
    project="market-sma-agent",
    base_model="Qwen/Qwen2.5-7B-Instruct",  # or your Unsloth 4-bit model
)

# To run on a T4, we need to override some config defaults.
model._internal_config = art.dev.InternalModelConfig(
    init_args=art.dev.InitArgs(
        max_seq_length=8192,
    ),
    engine_args=art.dev.EngineArgs(
        enforce_eager=True,
        gpu_memory_utilization=0.8,
    ),
)

# Initialize the server
backend = LocalBackend(
    # Normally we don't want to run the server in-process, but for the output
    # to show up properly on Google Colab we'll enable this.
    in_process=True,
    path="./.art",
)

# Register the model with the local Backend (sets up logging, inference, and training)
await model.register(backend)

/tmp/ipykernel_3752/809742586.py:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 08-05 14:21:15 [__init__.py:235] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: nicolepcx to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Initializing weave.
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:648: ArbitraryTypeWarning: <built-in function allocate_lock> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warnings.warn(
weave: Logged in as Weights & Biases user: nicolepcx.
weave: View Weave data at https://wandb.ai/nicolepcx/market-sma-agent/weave


INFO 08-05 14:21:30 [vllm_utils.py:689] Unsloth: Patching vLLM v1 graph capture
INFO 08-05 14:21:30 [vllm_utils.py:717] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.10.3: Fast Qwen2 patching. Transformers: 4.53.2. vLLM: 0.10.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit with actual GPU utilization = 78.54%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 79.25 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 8192. Num Sequences = 368.
Unsloth: vLLM's KV Cache can use up to 56.38 GB. Also swap space = 6 GB.
Unsloth: Not an error, 

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 08-05 14:21:59 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 08-05 14:22:01 [model_runner.py:1115] Model loading took 6.7339 GiB and 3.933356 seconds
INFO 08-05 14:22:05 [worker.py:295] Memory profiling takes 2.59 seconds
INFO 08-05 14:22:05 [worker.py:295] the current vLLM instance can use total_gpu_memory (79.25GiB) x gpu_memory_utilization (0.80) = 63.40GiB
INFO 08-05 14:22:05 [worker.py:295] model weights take 6.73GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 2.00GiB; the rest of the memory reserved for KV Cache is 54.57GiB.
INFO 08-05 14:22:05 [executor_base.py:113] # cuda blocks: 63864, # CPU blocks: 7021
INFO 08-05 14:22:05 [executor_base.py:118] Maximum concurrency for 8192 tokens per request: 124.73x
INFO 08-05 14:22:09 [llm_engine.py:424] init engine (profile, create kv cache, warmup model) took 7.57 seconds
Unsloth: Just some info: will skip parsing ['input_layernorm', 'norm2', 'q_norm', 'ffn_norm', 'post_feedforward_layernorm', 'layer

Unsloth 2025.10.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Scenarios

In [5]:
from pydantic import BaseModel

class Scenario(BaseModel):
    id: str
    question: str
    ticker: str
    window: int
    period_days: int
    # tol in final rounded space
    tolerance: float = 0.02

scenarios = [
    Scenario(
        id="aapl_sma_3",
        question="What is the 3 day SMA for AAPL as of the latest close? Return only the number rounded to 2 decimals.",
        ticker="AAPL",
        window=3,
        period_days=10,
        tolerance=0.02,
    ),
    Scenario(
        id="msft_sma_3",
        question="Compute the 3 day SMA for MSFT using the most recent closes. Return only the number rounded to 2 decimals.",
        ticker="MSFT",
        window=3,
        period_days=10,
        tolerance=0.02,
    ),
]


In [7]:
!pip install langchain_openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 101.1 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.90.0
    Uninstalling openai-1.90.0:
      Successfully uninstalled openai-1.90.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.23
    Uninstalling langchain-core-1.2.23:
      Successfully uninstalled langchain-core-1.2.23
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.10.0 requires openai<=1.90.0,>=1.87.0, but you have openai 2.53.0 which is incompatible.


# Rollout (LangGraph ReAct)

In [6]:
from art.langgraph import init_chat_model, wrap_rollout
import art

MAX_TURNS = 8

class ProjectTrajectory(art.Trajectory):
    final_answer: Optional[FinalAnswer] = None

class MarketScenario(BaseModel):
    step: int
    scenario: Scenario

@weave.op
async def rollout(model: art.Model, market_scenario: MarketScenario) -> ProjectTrajectory:
    scn = market_scenario.scenario

    # reset final holder
    nonlocal_final["value"] = None

    traj = ProjectTrajectory(
        reward=0.0,
        messages_and_choices=[],
        metadata={"scenario_id": scn.id, "ticker": scn.ticker, "window": scn.window},
    )

    system_prompt = dedent(f"""
        You are a market data agent.

        Use tools to fetch prices and compute the {scn.window}-day SMA:
        - Call latest_closes_tool(ticker="{scn.ticker}", n={scn.window}, period_days={scn.period_days})
        - Compute SMA = average of those closes.
        - Use calculator for exact arithmetic, e.g. '(x+y+z)/{scn.window}'
        - Round to 2 decimals.
        - MUST call return_final_answer_tool with ONLY the number string (e.g., "240.79").

        Constraints:
        - Do not guess. Always call latest_closes_tool first.
        - Do NOT add text; final answer tool gets only the number string.
    """)

    tools = [latest_closes_tool, calculator, return_final_answer_tool]
    chat = init_chat_model(model.name, temperature=0.2, max_tokens=200)
    agent = create_react_agent(chat, tools)

    try:
        config = {
            "configurable": {"thread_id": str(uuid.uuid4())},
            "recursion_limit": MAX_TURNS,
        }
        await agent.ainvoke(
            {"messages": [SystemMessage(content=system_prompt), HumanMessage(content=scn.question)]},
            config=config,
        )

        if nonlocal_final["value"]:
            traj.final_answer = nonlocal_final["value"]
            reward, truth, pred = judge_sma_numeric(
                scn.ticker, scn.window, scn.period_days, traj.final_answer.answer, tol=scn.tolerance
            )
            traj.reward = reward
            traj.metrics["truth"] = round(truth, 6) if truth == truth else None
            traj.metrics["pred"] = pred
            traj.metrics["error"] = (abs(round(truth,2)-pred) if truth==truth and pred==pred else None)
            traj.metrics["correct"] = float(reward)
    except Exception as e:
        print("Rollout error:", e)
        traj.messages_and_choices.append({"role": "assistant", "content": f"Error: {e}"})
    return traj


# Utils (printing summaries)

In [7]:
import pandas as pd

def groups_to_df(groups: list[art.TrajectoryGroup], label: str) -> pd.DataFrame:
    rows = []
    for gi, g in enumerate(groups):
        for ti, t in enumerate(g.trajectories):
            rows.append({
                "group": gi,
                "traj": ti,
                "reward": getattr(t, "reward", None),
                "correct": t.metrics.get("correct") if hasattr(t, "metrics") else None,
                "error": t.metrics.get("error") if hasattr(t, "metrics") else None,
                "truth": t.metrics.get("truth") if hasattr(t, "metrics") else None,
                "has_answer": bool(getattr(t, "final_answer", None)),
                "answer": getattr(getattr(t, "final_answer", None), "answer", None),
                "steps": len(getattr(t, "messages_and_choices", [])),
            })
    df = pd.DataFrame(rows)
    df.attrs["label"] = label
    return df

def print_group_summary(groups, title):
    print(f"\n=== {title} ===")
    df = groups_to_df(groups, title)
    if len(df):
        print(df.to_string(index=False))
    else:
        print("(no groups)")


# Training Loop (deterministic numeric reward)

In [8]:
from art.utils import iterate_dataset

training_config = {
    "groups_per_step": 1,
    "num_epochs": 2,
    "rollouts_per_group": 4,
    "learning_rate": 1e-5,
    "max_steps": 3,
}

training_iterator = iterate_dataset(
    scenarios,
    groups_per_step=training_config["groups_per_step"],
    num_epochs=training_config["num_epochs"],
    initial_step=await model.get_step(),
)

for batch in training_iterator:
    print(f"Training step {batch.step}, epoch {batch.epoch}")
    groups = []
    for scn in batch.items:
        group = art.TrajectoryGroup(
            [ wrap_rollout(model, rollout)(model, MarketScenario(step=batch.step, scenario=scn))
              for _ in range(training_config["rollouts_per_group"]) ]
        )
        groups.append(group)

    # Gather
    finished = await art.gather_trajectory_groups(
        groups, pbar_desc="gather",
        max_exceptions=training_config["rollouts_per_group"] * len(batch.items),
    )

    print_group_summary(finished, "Pre judge group summary")

    # We already computed numeric rewards in rollout
    judged = finished

    print_group_summary(judged, "Post judge group summary")

    await model.train(
        judged,
        config=art.TrainConfig(learning_rate=training_config["learning_rate"]),
        _config={"logprob_calculation_chunk_size": 8},
    )
    print(f"Completed training step {batch.step}")

    if batch.step >= training_config["max_steps"]:
        break


Iterating dataset:   0%|          | 0/4 [00:00<?, ?batch/s]

Training step 0, epoch 0


gather:   0%|          | 0/4 [00:00<?, ?it/s]

/tmp/ipykernel_3752/3301767379.py:43: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(chat, tools)
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy



=== Pre judge group summary ===
 group  traj  reward  correct  error      truth  has_answer answer  steps
     0     0     0.0      NaN    NaN        NaN       False   None      5
     0     1     0.0      0.0   0.04 307.041677        True 307.08      9
     0     2     0.0      0.0   0.04 307.041677        True 307.08      9
     0     3     0.0      0.0   0.04 307.041677        True 307.08      9

=== Post judge group summary ===
 group  traj  reward  correct  error      truth  has_answer answer  steps
     0     0     0.0      NaN    NaN        NaN       False   None      5
     0     1     0.0      0.0   0.04 307.041677        True 307.08      9
     0     2     0.0      0.0   0.04 307.041677        True 307.08      9
     0     3     0.0      0.0   0.04 307.041677        True 307.08      9


...

/tmp/ipykernel_3752/3301767379.py:43: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(chat, tools)
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy



=== Pre judge group summary ===
 group  traj  reward  correct  error     truth  has_answer answer  steps
     0     0     0.0      0.0   0.03 489.66333        True 489.69      9
     0     1     0.0      0.0   0.03 489.66333        True 489.69      9
     0     2     0.0      0.0   0.03 489.66333        True 489.69      9
     0     3     0.0      0.0   0.03 489.66333        True 489.69      9

=== Post judge group summary ===
 group  traj  reward  correct  error     truth  has_answer answer  steps
     0     0     0.0      0.0   0.03 489.66333        True 489.69      9
     0     1     0.0      0.0   0.03 489.66333        True 489.69      9
     0     2     0.0      0.0   0.03 489.66333        True 489.69      9
     0     3     0.0      0.0   0.03 489.66333        True 489.69      9
Skipping tuning as there is no suitable data. This can happen when all the trajectories in the same group have the same reward and thus no advantage to train on.
Advanced step from 2 to 3 (no training oc

gather:   0%|          | 0/4 [00:00<?, ?it/s]

/tmp/ipykernel_3752/3301767379.py:43: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(chat, tools)
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and wil


=== Pre judge group summary ===
 group  traj  reward  correct  error     truth  has_answer answer  steps
     0     0     1.0      1.0    0.0 307.09667        True 307.10      9
     0     1     1.0      1.0    0.0 307.09667        True 307.10      9
     0     2     1.0      1.0    0.0 307.09667        True 307.10      9
     0     3     1.0      1.0    0.0 307.09667        True 307.10      9

=== Post judge group summary ===
 group  traj  reward  correct  error     truth  has_answer answer  steps
     0     0     1.0      1.0    0.0 307.09667        True 307.10      9
     0     1     1.0      1.0    0.0 307.09667        True 307.10      9
     0     2     1.0      1.0    0.0 307.09667        True 307.10      9
     0     3     1.0      1.0    0.0 307.09667        True 307.10      9
Skipping tuning as there is no suitable data. This can happen when all the trajectories in the same group have the same reward and thus no advantage to train on.
Advanced step from 3 to 4 (no training oc

# Inference / Testing (prints the number only)

In [9]:
print("Testing trained model...\n")
test = MarketScenario(step=0, scenario=scenarios[0])
res = await wrap_rollout(model, rollout)(model, test)

if res.final_answer:
    # Print ONLY the numeric string (fulfills your exact output requirement)
    print(res.final_answer.answer)
else:
    print("No final answer.")


Testing trained model...



/tmp/ipykernel_3752/3301767379.py:43: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(chat, tools)
/tmp/ipykernel_3752/3869244591.py:24: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if hasattr(ast, "Num") and isinstance(node, ast.Num):  # legacy


307.09


# RULER judging path

In [11]:
from art.rewards import ruler_score_group

# After `finished = await art.gather_trajectory_groups(...)`:
judged_groups = []
for group in finished:
    judged_group = await ruler_score_group(group, "openai/gpt-5.4-mini-2026-03-17", debug=True)
    judged_groups.append(judged_group)

await model.train(
    judged_groups,
    config=art.TrainConfig(learning_rate=training_config["learning_rate"]),
    _config={"logprob_calculation_chunk_size": 8},
)


[RULER] Pretty-printed LLM choice JSON:

{
    'scores': [
        {
            'trajectory_id': '1',
            'explanation': 'Fetched the correct closes and computed the SMA correctly, but the final assistant 
message after using the final-answer tool is wrong/inconsistent with the computed result.',
            'score': 0.72
        },
        {
            'trajectory_id': '2',
            'explanation': 'Completed the required tool calls and returned the correct rounded SMA cleanly.',
            'score': 0.98
        },
        {
            'trajectory_id': '3',
            'explanation': 'Computed and returned the correct rounded SMA, but used a reference_ids field against 
the instruction and the final assistant message is malformed rather than simply ending with the answer.',
            'score': 0.9
        },
        {
            'trajectory_id': '4',
            'explanation': 'Fetched the closes and produced the correct rounded SMA, though it used a slightly 
different close set and therefore appears less exact than the others.',
            'score': 0.88
        }
    ]
}

Packed 4 trajectories into 1 sequences of length 2048


train:   0%|          | 0/1 [00:00<?, ?it/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000,000 | Num Epochs = 3 | Total steps = 30,000,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 20,185,088 of 7,635,801,600 (0.26% trained)


Unsloth: Will smartly offload gradients to save VRAM!
